In [ ]:
#A Install packages #1
!pip install transformers datasets peft accelerate sacrebleu sentencepiece


In [ ]:
#A Install packages #2
!pip install torchao==0.16.0
!pip install protobuf

In [ ]:
#A Install packages #3
!pip install evaluate sacrebleu rouge_score


In [ ]:
#B Try translation with MT5.#1
#V1
#Status: Worked with Tamil Token But failed with Actual Baduga Words. Token not in pretrained being the root cause of the problem

import os
import torch
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)
from peft import LoraConfig, TaskType, get_peft_model

# =====================================================================
# 1. SETUP & 5-ROW BADAGA (IN TAMIL SCRIPT) DATASET
# =====================================================================
# mT5-base provides robust vocabulary embeddings for Tamil Unicode blocks
MODEL_ID = "google/mt5-base"
SOURCE_KEY = "english"
TARGET_KEY = "badaga"
MAX_LENGTH = 64

# Your absolute 5-row python list of dicts (Badaga represented via Tamil script)
my_custom_data = [
    {"english": "I am here.", "badaga": "நா இல்லி இதே."},
    {"english": "you are here.", "badaga": "நீ இல்லி இதே"},
    {"english": "we are here.", "badaga": "நாங்க இல்லி இதோ"},
    {"english": "I am there", "badaga": "நா அல்லி இதே"},
    {"english": "You are there.", "badaga": "நீ அல்லி இதே"}
]

# Load list to Dataset.
# NOTE: Because we only have 5 rows, we skip splitting to avoid empty test validation tensors.
full_dataset = Dataset.from_list(my_custom_data)

print("Downloading and preparing Multilingual T5 Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

# =====================================================================
# 2. DATA TOKENIZATION & LOSS PROTECTION
# =====================================================================
def preprocess_function(examples):
    # Formulate prompts that declare the exact target minority language context
    inputs = [f"translate English to Badaga: {src.strip()}" for src in examples[SOURCE_KEY]]
    targets = [tgt.strip() for tgt in examples[TARGET_KEY]]

    model_inputs = tokenizer(inputs, max_length=MAX_LENGTH, padding="max_length", truncation=True)
    labels = tokenizer(text_target=targets, max_length=MAX_LENGTH, padding="max_length", truncation=True)
    print(labels)
    # Map pad tokens to -100 so the cross-entropy engine prevents blank emissions
    labels_ids = labels["input_ids"]
    cleaned_labels_ids = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label]
        for label in labels_ids
    ]

    model_inputs["labels"] = cleaned_labels_ids
    return model_inputs

tokenized_dataset = full_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=[SOURCE_KEY, TARGET_KEY]
)

# =====================================================================
# 3. INITIALIZE MULTILINGUAL BASE MODEL & LORA
# =====================================================================
print("\n--- Loading Pretrained mT5 Architecture ---")
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

# Configure LoRA Parameters for low resource text alignment
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,                             # Slightly elevated rank to capture specific phonemes
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q", "v"]         # Targets attention states
)

model = get_peft_model(base_model, peft_config)
model.print_trainable_parameters()

# =====================================================================
# 4. TRAINING WITH MODERN ARGUMENTS (OVER-FITTING FOR 5 SAMPLES)
# =====================================================================
training_args = Seq2SeqTrainingArguments(
    output_dir="./lora-badaga-run",
    eval_strategy="epoch",
    learning_rate=2e-3,               # High learning rate to enforce memorisation on tiny data
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=30,              # High epochs so 5 lines can settle inside the adapter weights
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_dataset,   # Pointing validation to train set to avoid empty evaluation errors
    processing_class=tokenizer,      # Cleaned keyword argument fix
    data_collator=data_collator,
)

print("\n--- Starting Fine-tuning Pipeline ---")
trainer.train()

# Save final local artifacts
model.save_pretrained("./badaga_lora_adapter")
tokenizer.save_pretrained("./badaga_lora_adapter")
print("Badaga Tamil-script adapters saved successfully.")



In [ ]:
#B Try translation with MT5.#2
#V1
# =====================================================================
# 6. INFERENCE TESTING ON TRAINED SENTENCES
# =====================================================================
print("\n--- Running Inference Testing ---")
unseen_samples = [
    "I am  here.",
    "we are here",
    "we are there. I am there"
]

for idx, text in enumerate(unseen_samples, 1):
    infer_prompt = f"translate English to Badaga: {text}"
    inputs = tokenizer(infer_prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=4,
            early_stopping=True
        )

    translated_text = tokenizer.decode(outputs, skip_special_tokens=True)
    print(f"\n[Inference Sample {idx}]")
    print(f"  Source (EN): {text}")
    print(f"  Result (Badaga/Tamil Script): {translated_text}")


In [ ]:
#B Try transaltion with MT5.#3
#V1
# =====================================================================
# 5. VALIDATION TESTING WITH EVALUATE ENGINE
# =====================================================================
print("\n--- Running Token Verification Performance Checks ---")
chrf_metric = evaluate.load("chrf")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

predictions = []
references = []

for item in full_dataset:
    src_text = item[SOURCE_KEY]
    ref_text = item[TARGET_KEY]

    prompt = f"translate English to Badaga: {src_text}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_LENGTH,
            num_beams=2,
            early_stopping=True
        )

    pred_text = tokenizer.decode(outputs, skip_special_tokens=True)
    predictions.append(pred_text)
    references.append([ref_text])

chrf_results = chrf_metric.compute(predictions=predictions, references=references)
print(f"Validation Overfit ChrF Score: {chrf_results['score']:.2f}")



In [ ]:
#C Try transaltion with NLLB.#1
#V1 Worked well with Tomil Tokens. Was able to cover more words.
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# 1. Define your custom dataset
my_custom_data = [
    {"english": "I am here.", "badaga": "நான் இங்கே இருக்கிறேன்."},
    {"english": "you are here.", "badaga": "நீங்கள் இங்கே இருக்கிறீர்கள்"},
    {"english": "we are here.", "badaga": "நாங்கள் இங்கே இருக்கிறோம்"},
    {"english": "I am there", "badaga": "நான் அங்கே இருக்கிறேன்"},
    {"english": "You are there.", "badaga": "நீங்கள் அங்கே இருக்கிறீர்கள்"},
]

# Convert to Hugging Face Dataset format
dataset = Dataset.from_list(my_custom_data)

# 2. Load NLLB Tokenizer and Model
# Using the distilled 600M parameter model for efficiency
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Set source and target language codes
# 'eng_Latn' for English. Using 'tam_Taml' (Tamil) as a proxy for Badaga script compatibility.
SRC_LANG = "eng_Latn"
TGT_LANG = "tam_Taml"

tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = TGT_LANG


# 3. Preprocess Data with padding capacity for longer sentences
def preprocess_function(examples):
    inputs = examples["english"]
    targets = examples["badaga"]

    # max_length=128 allows the model to handle much longer sentences in the future
    model_inputs = tokenizer(
        inputs, text_target=targets, max_length=128, truncation=True
    )
    return model_inputs


tokenized_dataset = dataset.map(preprocess_function, batched=True)

# 4. Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 5. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-badaga-translator",
    eval_strategy="no",  # Skipped due to small dataset size
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=4,  # Higher epochs needed for tiny datasets to learn mappings
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),  # Use mixed precision if GPU is available
    logging_steps=2,
)

# 6. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

# 7. Train and Save the Model
print("Starting training...")
trainer.train()
model.save_pretrained("./fine_tuned_badaga_model")
tokenizer.save_pretrained("./fine_tuned_badaga_model")
print("Model saved successfully!")


# =====================================================================
# 8. Inference: Test on a longer sentence containing trained words
# =====================================================================
def translate_english_to_badaga(text):
    # Load fine-tuned assets
    ft_tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_badaga_model")
    ft_model = AutoModelForSeq2SeqLM.from_pretrained(
        "./fine_tuned_badaga_model"
    )

    ft_tokenizer.src_lang = SRC_LANG

    # Tokenize input
    inputs = ft_tokenizer(text, return_tensors="pt", max_length=128, truncation=True)

    # Generate translation using the Badaga target token
    translated_tokens = ft_model.generate(
        **inputs,
        forced_bos_token_id=ft_tokenizer.convert_tokens_to_ids(TGT_LANG),
        max_length=128,
    )

    # Decode output
    result = ft_tokenizer.batch_decode(
        translated_tokens, skip_special_tokens=True
    )[0]
    return result


# Test with a longer sentence reconstructed from your vocabulary words
long_test_sentence = "I am here and we are there."
translation = translate_english_to_badaga(long_test_sentence)
print(f"\nInput: {long_test_sentence}")
print(f"Output Translation: {translation}")


In [12]:
#C Try translation with NLLB.#2
#V1 Worked well with Tomil Tokens. Was able to cover more words.
# Test with a longer sentence reconstructed from your vocabulary words
long_test_sentence = "I,We,You. Here,There"
translation = translate_english_to_badaga(long_test_sentence)
print(f"\nInput: {long_test_sentence}")
print(f"Output Translation: {translation}")


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]


Input: I,We,You. Here,There
Output Translation: நான், நாங்கள், நீங்கள் இங்கே, அங்கே


In [ ]:
#Another Option M2M100_418 M Placeholder


In [ ]:
#C Try translation with NLLB.#1
#V2  #Added Language Code.
     #Borrowing Tokens from another rare language pag_LATN,Adjusted Learning Rate,epochs
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

# 1. Define your custom dataset
my_custom_data = [
    {"english": "I am here.", "badaga": "Na Illi Idhae."},
    {"english": "you are here.", "badaga": "Ni Illi Idhae"},
    {"english": "we are here.", "badaga": "Naanga Illi Idho"},
    {"english": "I am there", "badaga": "Na Alli Idhae"},
    {"english": "You are there.", "badaga": "Ni Alli Idhae"},
]

dataset = Dataset.from_list(my_custom_data)

# 2. Load NLLB Tokenizer and Model
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# --- CRITICAL FIX FOR NEW LANGUAGE CODE ---
NEW_LANG = "pag_Latn"
SRC_LANG = "eng_Latn"

# 1. Explicitly add to special tokens so it's treated as a control token
tokenizer.add_special_tokens({"additional_special_tokens": [NEW_LANG]})

# 2. Update the tokenizer's internal language-to-id mappings
if hasattr(tokenizer, "lang_code_to_id"):
    tokenizer.lang_code_to_id[NEW_LANG] = tokenizer.convert_tokens_to_ids(NEW_LANG)

# 3. Resize model embedding layers to hold the new token
model.resize_token_embeddings(len(tokenizer))

# Set runtime language variables
tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = NEW_LANG


# 3. Preprocess Data
def preprocess_function(examples):
    inputs = examples["english"]
    targets = examples["badaga"]

    # Explicitly set tokenizer context for targets
    tokenizer.src_lang = SRC_LANG
    tokenizer.tgt_lang = NEW_LANG

    model_inputs = tokenizer(
        inputs, text_target=targets, max_length=128, truncation=True
    )
    return model_inputs


tokenized_dataset = dataset.map(preprocess_function, batched=True)

# 4. Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 5. Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-badaga-translator",
    eval_strategy="no",
    learning_rate=3e-4,             # Marginally higher to force new token embedding changes
    per_device_train_batch_size=2,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=30,            # Increased epochs so the model memorizes the 5 rule mappings
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
)

# 6. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

# 7. Train and Save the Model
print("Starting training...")
trainer.train()
model.save_pretrained("./fine_tuned_badaga_model")
tokenizer.save_pretrained("./fine_tuned_badaga_model")
print("Model saved successfully!")


# =====================================================================
# 8. Inference: Test Setup
# =====================================================================
def translate_english_to_badaga(text):
    # Load fine-tuned assets
    ft_tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_badaga_model")
    ft_model = AutoModelForSeq2SeqLM.from_pretrained("./fine_tuned_badaga_model")

    # Re-apply the language mapping structure upon fresh loading
    if hasattr(ft_tokenizer, "lang_code_to_id") and NEW_LANG not in ft_tokenizer.lang_code_to_id:
        ft_tokenizer.lang_code_to_id[NEW_LANG] = ft_tokenizer.convert_tokens_to_ids(NEW_LANG)

    ft_tokenizer.src_lang = SRC_LANG

    # Tokenize input
    inputs = ft_tokenizer(text, return_tensors="pt", max_length=128, truncation=True)

    # Fetch the exact registered internal token ID
    forced_bos_token_id = ft_tokenizer.convert_tokens_to_ids(NEW_LANG)

    # Generate translation using the Badaga target token
    translated_tokens = ft_model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=128,
    )

    # Decode output
    result = ft_tokenizer.batch_decode(
        translated_tokens, skip_special_tokens=True
    )[0]
    return result


# Test execution
long_test_sentence = "We are there.I am here"
translation = translate_english_to_badaga(long_test_sentence)
print(f"\nInput: {long_test_sentence}")
print(f"Output Translation: {translation}")


In [36]:
#C Try translation with NLLB.#2
#V2  #Added Language Code.
# Test execution. Overfitting, Inaccurate
long_test_sentence = "we are there"
translation = translate_english_to_badaga(long_test_sentence)
print(f"\nInput: {long_test_sentence}")
print(f"Output Translation: {translation}")


Loading weights:   0%|          | 0/509 [00:00<?, ?it/s]


Input: we are there
Output Translation: Naanga Illi Idho


In [ ]:
#C Try translation with NLLB
#V3
# accomodate more words, pick right words
import torch
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType

# 1. EXPANDED DATASET (Critical: Teaching the model individual words + combinations)
# If you only feed it full sentences, it cannot split them. We must feed it fragments.
my_custom_data = [
    # Original Sentences
    {"english": "I am here.", "badaga": "Na Illi Idhae."},
    {"english": "you are here.", "badaga": "Ni Illi Idhae"},
    {"english": "we are here.", "badaga": "Naanga Illi Idho"},
    {"english": "I am there", "badaga": "Na Alli Idhae"},
    {"english": "You are there.", "badaga": "Ni Alli Idhae"},

    # Word-Level Alignment Anchors (Forces the model to map individual tokens correctly)
    {"english": "I", "badaga": "Na"},
    {"english": "you", "badaga": "Ni"},
    {"english": "we", "badaga": "Naanga"},
    {"english": "here", "badaga": "Illi"},
    {"english": "there", "badaga": "Alli"},
    {"english": "and", "badaga": "battu"}, # Added a connector so it can handle longer links

    # Structural Variations (Teaches the model how to stitch them together)
    {"english": "I am here and you are there.", "badaga": "Na Illi Idhae battu ni alli idhae."},
    {"english": "We are here.", "badaga": "Naanga Illi Idho."}
]

dataset = Dataset.from_list(my_custom_data)

# 2. Load NLLB Tokenizer and Base Model
model_name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Using 'pag_Latn' as a stable internal routing proxy
SRC_LANG = "eng_Latn"
PROXY_TGT_LANG = "pag_Latn"

tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = PROXY_TGT_LANG

# 3. APPLY LoRA (Prevents the model from outputting scrambled tokens)
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=16,                  # Bottleneck dimensionality
    lora_alpha=32,         # Scaling factor
    lora_dropout=0.1,      # Regularization to prevent strict memorization
    target_modules=["q_proj", "v_proj"] # Target attention weights
)
model = get_peft_model(model, peft_config)
print("LoRA adapter successfully injected into NLLB.")

# 4. Preprocess Dataset
def preprocess_function(examples):
    inputs = examples["english"]
    targets = examples["badaga"]
    tokenizer.src_lang = SRC_LANG
    tokenizer.tgt_lang = PROXY_TGT_LANG
    return tokenizer(inputs, text_target=targets, max_length=128, truncation=True)

tokenized_dataset = dataset.map(preprocess_function, batched=True)
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

# 5. Training Arguments Optimized for Generalization
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-badaga-lora",
    eval_strategy="no",
    learning_rate=5e-4,             # Higher learning rate is perfect for LoRA training
    per_device_train_batch_size=2,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=120,           # Stable convergence threshold for LoRA with small text
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=10,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Starting LoRA fine-tuning...")
trainer.train()

# Save the adapter weights and tokenizer
model.save_pretrained("./fine_tuned_badaga_model")
tokenizer.save_pretrained("./fine_tuned_badaga_model")
print("Model saved successfully!")


# =====================================================================
# 6. Optimized Inference Engine (Fixes long sentence generation flaws)
# =====================================================================
def translate_english_to_badaga(text):
    from peft import PeftModel, PeftConfig

    # Reload base model and attach our newly trained adapter explicitly
    base_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
    ft_model = PeftModel.from_pretrained(base_model, "./fine_tuned_badaga_model")
    ft_tokenizer = AutoTokenizer.from_pretrained("./fine_tuned_badaga_model")

    ft_tokenizer.src_lang = SRC_LANG
    inputs = ft_tokenizer(text, return_tensors="pt", max_length=128, truncation=True)
    forced_bos_token_id = ft_tokenizer.convert_tokens_to_ids(PROXY_TGT_LANG)

    # ADVANCED GENERATION PARAMETERS: Prevents token loops and forces fluid mapping
    translated_tokens = ft_model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_length=128,
        num_beams=4,                 # Beam search checks multiple translation paths
        no_repeat_ngram_size=2,      # Absolutely blocks repeating token loops
        early_stopping=True
    )

    result = ft_tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]
    return result


# Test execution with a longer, combined sentence
long_test_sentence = "I am here and you are there."
translation = translate_english_to_badaga(long_test_sentence)
print(f"\nInput: {long_test_sentence}")
print(f"Output Translation: {translation}")


In [11]:
#C Try translation with NLLB.#2
#V3  can cover more words. need to adjust learning rate and epochs
# Test execution.
long_test_sentence = "we are there and you are here"
translation = translate_english_to_badaga(long_test_sentence)
print(f"\nInput: {long_test_sentence}")
print(f"Output Translation: {translation}")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]


Input: we are there and you are here
Output Translation: Na Alli Idhae battu ni alli idhae


In [4]:
#B Try translation with MT5.#1
#V3
#Status: Worked with Tamil Token
import torch
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, MT5ForConditionalGeneration

# ==========================================
# 1. SETUP DATASET AND VOCABULARY
# ==========================================
badaga_vocabulary = ["நா", "நீ", "நாங்க", "இல்லி", "அல்லி", "இதே", "இதோ"]

my_custom_data = [
    {"english": "translate English to Badaga: I am here.", "badaga": "நா இல்லி இதே."},
    {"english": "translate English to Badaga: you are here.", "badaga": "நீ இல்லி இதே"},
    {"english": "translate English to Badaga: we are here.", "badaga": "நாங்க இல்லி இதோ"},
    {"english": "translate English to Badaga: I am there", "badaga": "நா அல்லி இதே"},
    {"english": "translate English to Badaga: You are there.", "badaga": "நீ அல்லி இதே"}
]

# Set device context
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running execution loop on engine target: {device.upper()}")

# ==========================================
# 2. ACCURATE TOKENIZATION AND MASKING
# ==========================================
tokenizer = AutoTokenizer.from_pretrained("google/mt5-base", use_fast=True)
num_added_toks = tokenizer.add_tokens(badaga_vocabulary)

# Tokenize full structural batches using a compact matrix space
input_encodings = tokenizer([d["english"] for d in my_custom_data], padding=True, truncation=True, return_tensors="pt")
target_encodings = tokenizer([d["badaga"] for d in my_custom_data], padding=True, truncation=True, return_tensors="pt")

# Convert padding positions directly to -100 to completely remove them from loss calculations
labels = target_encodings["input_ids"].clone()
labels[labels == tokenizer.pad_token_id] = -100

# Push tensors to memory target
input_ids = input_encodings["input_ids"].to(device)
attention_mask = input_encodings["attention_mask"].to(device)
labels = labels.to(device)

# ==========================================
# 3. CONFIGURE INTERNALS AND OPTIMIZER
# ==========================================
model = MT5ForConditionalGeneration.from_pretrained("google/mt5-base")
if num_added_toks > 0:
    model.resize_token_embeddings(len(tokenizer))
    model.tie_weights()

model.to(device)

# Set model to training mode
model.train()

# Optimize ONLY the embedding layers to avoid breaking the core model structure
for param in model.parameters():
    param.requires_grad = False

model.shared.weight.requires_grad = True
model.lm_head.weight.requires_grad = True

# Standard learning rate for isolated embedding arrays
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

# ==========================================
# 4. NATIVE TRAINING LOOP (100 EPOCHS)
# ==========================================
print("\n--- Starting Isolated PyTorch Gradient Decoupling ---")
for epoch in range(1, 101):
    optimizer.zero_grad()

    # Forward pass calculates token-level cross-entropy loss natively
    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    loss = outputs.loss

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch}/100 | True Mathematical Mean Loss: {loss.item():.4f}")

print("Optimization complete!\n")

# ==========================================
# 5. INFERENCE AND VERIFICATION
# ==========================================
model.eval()

def translate_to_badaga(text):
    input_prompt = "translate English to Badaga: " + text
    inputs = tokenizer(input_prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs["input_ids"],
            max_length=16,
            num_beams=4,
            early_stopping=True
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=== RUNNING MEMORIZATION INFERENCE TESTS ===")
test_phrases = ["we are there.", "I am there", "you are here."]
for phrase in test_phrases:
    print(f"Input: '{phrase}' -> Predicted Badaga: {translate_to_badaga(phrase)}")


Running execution loop on engine target: CPU


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.



--- Starting Isolated PyTorch Gradient Decoupling ---
Epoch 1/100 | True Mathematical Mean Loss: 1472.5482
Epoch 10/100 | True Mathematical Mean Loss: 1626.5222
Epoch 20/100 | True Mathematical Mean Loss: 1708.8768
Epoch 30/100 | True Mathematical Mean Loss: 1622.4673
Epoch 40/100 | True Mathematical Mean Loss: 1624.5195
Epoch 50/100 | True Mathematical Mean Loss: 1740.7816
Epoch 60/100 | True Mathematical Mean Loss: 1523.1727
Epoch 70/100 | True Mathematical Mean Loss: 1643.7573
Epoch 80/100 | True Mathematical Mean Loss: 1444.7462
Epoch 90/100 | True Mathematical Mean Loss: 1862.2621
Epoch 100/100 | True Mathematical Mean Loss: 1912.5133
Optimization complete!

=== RUNNING MEMORIZATION INFERENCE TESTS ===
Input: 'we are there.' -> Predicted Badaga: 01.01.201ksuksuksuksuksu 01.01.201 01.01.201AnatomAnatomksuksuksuksuksu
Input: 'I am there' -> Predicted Badaga: 01.01.201밡ksuksuksuksuksuksuksuAnatomAnatomksuksuAnatomksu
Input: 'you are here.' -> Predicted Badaga: 01.01.201ksusqftsqftsq

In [5]:
#B Try translation with MT5.#2
#V3
#Status: Worked with Tamil Token
test_phrase = "we are there.I,we,you. here,there. I am here"
print(f"English Input: {test_phrase}")
print(f"Badaga Output: {translate_to_badaga(test_phrase)}")

English Input: we are there.I,we,you. here,there. I am here
Badaga Output: 01.01.201întâlnîntâlnîntâlnksu수있수있수있수있수있수있수있수있수있수있
